In [4]:
import pandas as pd
from google.cloud import bigquery


In [3]:
client = bigquery.Client()

DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.

In [ ]:
query = f"""
WITH prog AS (
  SELECT user_id 
        , dt 
        , min(IF(n_sets_completed = 12, dt, NULL)) OVER (PARTITION BY user_id) AS min_completed_dt 
  FROM `trailmixgames-game-1.merger_prod_fact.fact_dse_user_timed_album_progression_cumulative`
  WHERE dt >= '2025-11-03'
  and liveops_iteration_id = '202511031000'
)
, segments AS (
  SELECT user_id 
       , dt 
       , payment_value_28d_segment
  FROM `trailmixgames-game-1.merger_prod_fact.fact_dt_user_segments`
  WHERE dt >= '2025-11-03'
  AND active = 1 
)
, metrics AS (
  SELECT user_id 
       , dt 
       , active 
       , iap_rev
       , gems_spent 
       , gems_spent_minus_earned_game
       , gems_balance_end
  FROM `trailmixgames-game-1.merger_prod_ab.ab_dt_user_active_metrics`
  WHERE dt >= '2025-11-03'
)
  SELECT prog.dt 
       , COALESCE(min_completed_dt, '9999-09-09') AS min_completed_dt
       , DATE_DIFF(prog.dt, min_completed_dt, day) AS days_since_completed
      -- , payment_value_28d_segment
       , COUNT(DISTINCT prog.user_id) AS n_users_completed
       , SUM(COALESCE(active , 0)) AS active 
       , SUM(COALESCE(iap_rev, 0)) AS iap_rev
       , SUM(COALESCE(gems_spent, 0)) AS gems_spent 
       , SUM(COALESCE(gems_spent_minus_earned_game, 0)) AS gems_spent_minus_earned_game
  FROM prog 
  LEFT JOIN segments on prog.user_id = segments.user_id AND prog.min_completed_dt = segments.dt 
  LEFT JOIN metrics on prog.user_id = metrics.user_id AND prog.dt = metrics.dt 
  GROUP BY ALL 
  HAVING n_users_completed > 100
"""

df = client.query(query).to_dataframe()